# 01 — EDGAR Filing Exploration

Explore raw SEC EDGAR data before ingesting into the pipeline.

In [1]:
import sys
sys.path.insert(0, '..')
from app.ingestion.edgar_client import EdgarClient

client = EdgarClient()
cik = client.get_cik('AAPL')
print(f'Apple CIK: {cik}')

2026-04-24 21:09:05 | INFO     | app.ingestion.edgar_client | Resolved AAPL → CIK 0000320193
Apple CIK: 0000320193


In [2]:
filings = client.get_filing_urls('AAPL', form_type='10-K', max_filings=3)
for f in filings:
    print(f['filing_date'], f['form_type'], f['primary_doc_url'][:80])

2026-04-24 21:09:06 | INFO     | app.ingestion.edgar_client | Resolved AAPL → CIK 0000320193
2026-04-24 21:09:07 | INFO     | app.ingestion.edgar_client | Found 3 10-K filings for AAPL
2025-10-31 10-K https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.
2024-11-01 10-K https://www.sec.gov/Archives/edgar/data/320193/000032019324000123/aapl-20240928.
2023-11-03 10-K https://www.sec.gov/Archives/edgar/data/320193/000032019323000106/aapl-20230930.


In [3]:
# Download and inspect a single filing
if filings:
    path = client.download_filing(filings[0])
    print('Downloaded to:', path)
    content = path.read_text(errors='ignore')[:2000]
    print(content)

2026-04-24 21:09:07 | INFO     | app.ingestion.edgar_client | Already downloaded: C:\Users\dipes\Desktop\Projects\financial-rag-copilot\data\raw\filings\AAPL\10-K\AAPL_10-K_2025-10-31.htm
Downloaded to: C:\Users\dipes\Desktop\Projects\financial-rag-copilot\data\raw\filings\AAPL\10-K\AAPL_10-K_2025-10-31.htm
<?xml version='1.0' encoding='ASCII'?>
<!--XBRL Document Created with the Workiva Platform-->
<!--Copyright 2025 Workiva-->
<!--r:b93d322a-f6d3-4356-8a13-e3e1c42e12bd,g:44705cc3-5a3a-440a-975b-ed1d3694d858,d:719388195b384d85a4e238ad88eba90a-->
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:dei="http://xbrl.sec.gov/dei/2025" xmlns:link="http://www.xbrl.org/2003/linkbase" xmlns:country="http://xbrl.sec.gov/country/2025" xmlns:ixt="http://www.xbrl.org/inlineXBRL/transformation/2020-02-12" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:aapl="http://www.apple.com/20250927" xmlns:xbrli="http://www.xbrl.org/2003/instance" xmlns:xbrldi="http://xbrl.org/2006/xbrldi" xmlns:xlin

In [4]:
# Parse sections
from app.ingestion.parser import FilingParser
parser = FilingParser()
if filings:
    sections = parser.parse_file(path)
    for s in sections:
        print(f"[{s['section_name']}] {len(s['text'])} chars")

2026-04-24 21:09:07 | INFO     | app.ingestion.parser | Parsed 15 sections from AAPL_10-K_2025-10-31.htm
[Preamble] 19660 chars
[Quantitative and Qualitative Disclosures About Market Risk] 2829 chars
[Business] 16051 chars
[Risk Factors] 79550 chars
[Management's Discussion and Analysis] 18013 chars
[Quantitative and Qualitative Disclosures About Market Risk] 2956 chars
[Financial Statements] 638 chars
[Notes to Financial Statements] 1614 chars
[Notes to Financial Statements] 1112 chars
[Notes to Financial Statements] 1668 chars
[Notes to Financial Statements] 1358 chars
[Notes to Financial Statements] 2327 chars
[Notes to Financial Statements] 43 chars
[Notes to Financial Statements] 60829 chars
[Notes to Financial Statements] 13103 chars
